# Evaluación del enrutamiento del agente

Hasta acá la validación cuantitativa cubría dos piezas: el **retrieval** (notebooks 02, 03 y 08) y la **generación del LLM sobre un contexto fijo** (notebook 07). La corrección del docente señaló el hueco: *del agente completo solo medimos esas dos partes; el `router` no se mide*.

Este notebook cierra ese hueco midiendo **únicamente la decisión del router**: dada una consulta en lenguaje natural, ¿elige el especialista correcto entre los tres dominios activos?

La metodología es la misma que en los notebooks anteriores:

1. **Ground truth propio** en `tests/` (30 consultas etiquetadas, 10 por dominio).
2. **Dataset en LangSmith** con `inputs=pregunta` y `outputs=dominio_esperado`.
3. **`evaluate()` de LangSmith** con un evaluador automático de acierto.
4. **Resultados crudos en `reports/`**, resumen con IC 95 %, matriz de confusión y prueba pareada de McNemar entre dos modelos.

> No se invocan tools, no se consulta FAISS y no se genera la respuesta final: si el router elige mal, el resto del grafo ni llega a ejecutarse. Medir el router aislado permite atribir el error a esa decisión y no a las herramientas.

## 1. Qué se mide y qué no

El nodo `router` de `src/capazeta/graph/nodes.py` recibe el mensaje del usuario, usa `ROUTER_SYSTEM_PROMPT` y devuelve una clave de `DOMAINS`. Esa clave dispara la arista condicional hacia el subgrafo correspondiente.

**Se mide:**

- **Acierto de enrutamiento:** `dominio_predicho == dominio_esperado` (métrica principal).
- **Validez del formato:** que la salida cruda del router sea exactamente una de las tres claves. Importa porque `_extract_domain()` tiene un *fallback silencioso* a `threat_intel`: sin esta métrica, una respuesta inválida podría disfrazarse de acierto.
- **Latencia de la decisión** y matrices de confusión por modelo.

**No se mide:** ejecución de Nmap/Wazuh, calidad del RAG, fidelidad de la respuesta final ni consumo de tokens del subgrafo elegido.

Se evalúan dos modelos del catálogo del `.env`: el activo (`MODEL_1`, Gemini) y un alternativo (`MODEL_2`, Groq). Ambos reciben las **mismas 30 preguntas**, con `temperature=0` y la semilla fija del proyecto, en una sola corrida (`num_repetitions=1`).

In [1]:
import os
import time
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, SystemMessage
from langsmith import Client
from langsmith.evaluation import evaluate

from capazeta.config import LLM_SEED, LLM_TEMPERATURE
from capazeta.graph.nodes import (
    DOMAINS,
    ROUTER_SYSTEM_PROMPT,
    _extract_domain,
    _message_text,
)
from capazeta.llm import _seed_kwargs

ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()
)
TESTS = ROOT / "tests"
REPORTS = ROOT / "reports"
load_dotenv(ROOT / ".env")

MODELOS = {
    "gemini": os.getenv("MODEL_1") or "google_genai:gemini-3.1-flash-lite",
    "groq": os.getenv("MODEL_2") or "groq:openai/gpt-oss-120b",
}
CLASES = list(DOMAINS)

print("Raíz del repo:", ROOT)
print("Dominios activos:", CLASES)
print("Temperatura:", LLM_TEMPERATURE, "| semilla:", LLM_SEED)
for nombre, modelo in MODELOS.items():
    print(f"  router {nombre}: {modelo}")

Raíz del repo: /home/joaquincernik/Desktop/ia/final
Dominios activos: ['infraestructura_redes', 'soc_logs', 'threat_intel']
Temperatura: 0.0 | semilla: 42
  router gemini: google_genai:gemini-3.1-flash-lite
  router groq: groq:openai/gpt-oss-120b


In [2]:
gt = pd.read_csv(TESTS / "dataset_evaluacion_routing.csv")
conteos = gt["dominio_esperado"].value_counts().to_dict()

assert len(gt) == 30, f"Se esperaban 30 consultas y llegaron {len(gt)}"
assert conteos == {clase: 10 for clase in CLASES}, f"Dataset desbalanceado: {conteos}"
assert set(gt["dominio_esperado"]) <= set(CLASES), "Hay etiquetas fuera de DOMAINS"
assert gt["pregunta"].is_unique, "Hay preguntas duplicadas"

print(f"Ground truth: {len(gt)} consultas, {len(CLASES)} clases, 10 por clase")
print("Columnas:", list(gt.columns))
gt.head(6)

Ground truth: 30 consultas, 3 clases, 10 por clase
Columnas: ['pregunta', 'dominio_esperado', 'notas']


,pregunta,dominio_esperado,notas
0,¿Cómo debería segmentar una red corporativa co...,infraestructura_redes,NaN
1,Tenemos un firewall con reglas duplicadas y qu...,infraestructura_redes,NaN
2,¿Qué configuración de VPN conviene para usuari...,infraestructura_redes,NaN
3,El enlace entre sucursales tiene mucha latenci...,infraestructura_redes,NaN
4,Necesito diseñar la topología de red de un nue...,infraestructura_redes,NaN
5,¿Cómo verifico que el switch de acceso tenga e...,infraestructura_redes,NaN


## 2. Pipeline: el router aislado

En vez de reescribir el router, se importan **los objetos de producción** (`ROUTER_SYSTEM_PROMPT`, `_extract_domain` y `_message_text`). Cambia únicamente la instancia del modelo: se construye con los mismos argumentos que `get_llm()` (`temperature` y `_seed_kwargs`), uno por proveedor.

Cada corrida devuelve, además de la clase predicha, el **texto crudo** del router para poder auditar los fallos de formato. Si una llamada falla, la fila se marca con `error` y se puntúa como fallo en vez de detener el experimento.

In [3]:
def construir_router(modelo: str):
    """Misma configuración que get_llm(), pero con el modelo que pidamos."""
    proveedor = modelo.split(":", 1)[0].strip()
    return init_chat_model(
        modelo, temperature=LLM_TEMPERATURE, **_seed_kwargs(proveedor)
    )


def factory_pipeline(modelo: str, router):
    def pipeline(inputs: dict) -> dict:
        pregunta = inputs["pregunta"]
        inicio = time.perf_counter()
        try:
            mensaje = router.invoke(
                [
                    SystemMessage(content=ROUTER_SYSTEM_PROMPT),
                    HumanMessage(content=pregunta),
                ]
            )
            texto = _message_text(mensaje).strip()
            return {
                "dominio_predicho": _extract_domain(texto),
                "respuesta_router": texto,
                "formato_valido": int(texto.lower() in DOMAINS),
                "modelo": modelo,
                "latencia_s": round(time.perf_counter() - inicio, 3),
                "error": "",
            }
        except Exception as exc:
            return {
                "dominio_predicho": "",
                "respuesta_router": "",
                "formato_valido": 0,
                "modelo": modelo,
                "latencia_s": round(time.perf_counter() - inicio, 3),
                "error": str(exc)[:300],
            }

    return pipeline


routers = {nombre: construir_router(modelo) for nombre, modelo in MODELOS.items()}
pipelines = {
    nombre: factory_pipeline(modelo, routers[nombre]) for nombre, modelo in MODELOS.items()
}

print("Smoke test · una pregunta por clase (6 llamadas):")
for nombre, pipeline in pipelines.items():
    for clase in CLASES:
        pregunta = gt.loc[gt["dominio_esperado"] == clase, "pregunta"].iloc[0]
        salida = pipeline({"pregunta": pregunta})
        print(
            f"  [{nombre:6s}] esperado={clase:20s} "
            f"predicho={salida['dominio_predicho']:20s} "
            f"texto={salida['respuesta_router'][:28]!r}"
        )

Smoke test · una pregunta por clase (6 llamadas):


  [gemini] esperado=infraestructura_redes predicho=infraestructura_redes texto='infraestructura_redes'


  [gemini] esperado=soc_logs             predicho=soc_logs             texto='soc_logs'


  [gemini] esperado=threat_intel         predicho=threat_intel         texto='threat_intel'


  [groq  ] esperado=infraestructura_redes predicho=infraestructura_redes texto='infraestructura_redes'


  [groq  ] esperado=soc_logs             predicho=soc_logs             texto='soc_logs'


  [groq  ] esperado=threat_intel         predicho=threat_intel         texto='threat_intel'


## 3. Evaluador y dataset en LangSmith

El evaluador es determinista: compara la etiqueta de referencia con la clase devuelta por el pipeline y reporta `acierto`, `formato_valido` y `latencia_s` como feedback de LangSmith. Es el mismo patrón de los evaluadores de Hit@5/MRR de los notebooks 02 y 08.

El dataset se crea una sola vez y se reutiliza si ya existe con 30 ejemplos; si existe con otra cantidad, la celda se detiene para no duplicar datos (idempotencia igual a la del notebook 07).

In [4]:
def routing_evaluator(run, example) -> list[dict]:
    """Evaluador automático: acierto, formato y latencia de la decisión."""
    esperado = str((example.outputs or {}).get("dominio_esperado", "")).strip()
    salida = run.outputs or {}
    predicho = str(salida.get("dominio_predicho", "")).strip()
    return [
        {"key": "acierto", "score": float(predicho == esperado)},
        {"key": "formato_valido", "score": float(bool(salida.get("formato_valido", 0)))},
        {"key": "latencia_s", "score": float(salida.get("latencia_s") or 0.0)},
    ]


DATASET_NAME = "Evaluacion_Routing_Capazeta"
client = Client()

ejemplos = []
for fila in gt.itertuples(index=False):
    notas = "" if pd.isna(fila.notas) else str(fila.notas)
    ejemplos.append(
        {
            "inputs": {"pregunta": fila.pregunta},
            "outputs": {
                "dominio_esperado": fila.dominio_esperado,
                "notas": notas,
            },
        }
    )

if client.has_dataset(dataset_name=DATASET_NAME):
    dataset = client.read_dataset(dataset_name=DATASET_NAME)
    existentes = len(list(client.list_examples(dataset_id=dataset.id)))
    if existentes != len(ejemplos):
        raise ValueError(
            f"El dataset '{DATASET_NAME}' ya existe con {existentes} ejemplos y "
            f"se esperaban {len(ejemplos)}. Revisalo en LangSmith antes de continuar."
        )
    print(f"Dataset '{DATASET_NAME}' reutilizado ({existentes} ejemplos).")
else:
    dataset = client.create_dataset(
        dataset_name=DATASET_NAME,
        description=(
            "30 consultas de ciberseguridad con el dominio activo esperado, para "
            "evaluar el router de Capazeta (3 clases balanceadas)."
        ),
    )
    client.create_examples(dataset_id=dataset.id, examples=ejemplos)
    print(f"Dataset '{DATASET_NAME}' creado con {len(ejemplos)} ejemplos.")

Dataset 'Evaluacion_Routing_Capazeta' creado con 30 ejemplos.


## 4. Experimentos en LangSmith

Dos corridas independentes sobre el mismo dataset, una por modelo, con `max_concurrency=2` para no saturar las APIs. Cada `evaluate()` devuelve un objeto con `.to_pandas()`; se guarda tal cual en `reports/` para que los recursos sean auditables.

In [5]:
resultados_gemini = evaluate(
    pipelines["gemini"],
    data=DATASET_NAME,
    evaluators=[routing_evaluator],
    experiment_prefix="Capazeta_Routing_Gemini",
    description="Router de Capazeta con el modelo activo (MODEL_1, Gemini).",
    num_repetitions=1,
    max_concurrency=2,
)
df_gemini_raw = resultados_gemini.to_pandas()
ruta_gemini = REPORTS / "evaluacion_routing_gemini_raw.csv"
df_gemini_raw.to_csv(ruta_gemini, index=False, encoding="utf-8-sig")
print(f"Gemini: {len(df_gemini_raw)} filas guardadas en {ruta_gemini.name}")
print("Columnas del export:", df_gemini_raw.columns.tolist())

View the evaluation results for experiment: 'Capazeta_Routing_Gemini-7c02ec48' at:
https://smith.langchain.com/o/954863b9-6bd1-4509-9100-6980d0c6d6e0/datasets/9547f84a-cd4a-49d8-abff-e146f65f50cc/compare?selectedSessions=2b3842c3-8bb3-497d-a133-68a372563e53




Gemini: 30 filas guardadas en evaluacion_routing_gemini_raw.csv
Columnas del export: ['inputs.pregunta', 'outputs.dominio_predicho', 'outputs.respuesta_router', 'outputs.formato_valido', 'outputs.modelo', 'outputs.latencia_s', 'outputs.error', 'error', 'reference.notas', 'reference.dominio_esperado', 'feedback.acierto', 'feedback.formato_valido', 'feedback.latencia_s', 'execution_time', 'example_id', 'id']


In [6]:
resultados_groq = evaluate(
    pipelines["groq"],
    data=DATASET_NAME,
    evaluators=[routing_evaluator],
    experiment_prefix="Capazeta_Routing_Groq",
    description="Router de Capazeta con el modelo alternativo (MODEL_2, Groq).",
    num_repetitions=1,
    max_concurrency=2,
)
df_groq_raw = resultados_groq.to_pandas()
ruta_groq = REPORTS / "evaluacion_routing_groq_raw.csv"
df_groq_raw.to_csv(ruta_groq, index=False, encoding="utf-8-sig")
print(f"Groq: {len(df_groq_raw)} filas guardadas en {ruta_groq.name}")
print("Columnas del export:", df_groq_raw.columns.tolist())

View the evaluation results for experiment: 'Capazeta_Routing_Groq-15de9141' at:
https://smith.langchain.com/o/954863b9-6bd1-4509-9100-6980d0c6d6e0/datasets/9547f84a-cd4a-49d8-abff-e146f65f50cc/compare?selectedSessions=70fd043b-d13f-4fbc-90d8-fc92bbc49e6c




Groq: 30 filas guardadas en evaluacion_routing_groq_raw.csv
Columnas del export: ['inputs.pregunta', 'outputs.dominio_predicho', 'outputs.respuesta_router', 'outputs.formato_valido', 'outputs.modelo', 'outputs.latencia_s', 'outputs.error', 'error', 'reference.notas', 'reference.dominio_esperado', 'feedback.acierto', 'feedback.formato_valido', 'feedback.latencia_s', 'execution_time', 'example_id', 'id']


## 5. Resumen por modelo

Celda autosuficiente: lee los CSVs crudos de `reports/`, así que se puede correr aunque se reinicie el kernel. Reporta:

- **Accuracy** con **IC 95 % de Wilson** (más adecuado que el intervalo normal para proporciones con n = 30).
- **Precisión, recall y F1 macro** sobre las tres clases (`zero_division=0`).
- **Matriz de confusión** 3×3 (más una categoría `<inválido>` para las salidas que no son una clave).
- **% de formato válido** y **latencia mediana** de la decisión.

Archivos: `reports/evaluacion_routing_resumen.csv` y `reports/evaluacion_routing_matriz_<modelo>.csv`.

In [7]:
from pathlib import Path

import pandas as pd
from sklearn.metrics import precision_recall_fscore_support

if "REPORTS" not in globals():
    ROOT = next(
        p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()
    )
    REPORTS = ROOT / "reports"
if "CLASES" not in globals():
    from capazeta.graph.nodes import DOMAINS

    CLASES = list(DOMAINS)
if "MODELOS" not in globals():
    MODELOS = {"gemini": "gemini", "groq": "groq"}


def _columna(tabla: pd.DataFrame, *candidatas: str) -> pd.Series:
    for nombre in candidatas:
        if nombre in tabla.columns:
            return tabla[nombre]
    raise KeyError(f"No encontré ninguna de {candidatas} en {list(tabla.columns)}")


def cargar_routing(nombre: str) -> pd.DataFrame:
    """Normaliza el export de LangSmith de una corrida de routing."""
    tabla = pd.read_csv(REPORTS / f"evaluacion_routing_{nombre}_raw.csv")
    return pd.DataFrame(
        {
            "pregunta": _columna(tabla, "inputs.pregunta", "pregunta").astype(str),
            "dominio_esperado": _columna(
                tabla, "reference.dominio_esperado", "outputs.dominio_esperado", "dominio_esperado"
            ).astype(str),
            "dominio_predicho": _columna(
                tabla, "outputs.dominio_predicho", "dominio_predicho"
            ).fillna("").astype(str),
            "acierto": pd.to_numeric(
                _columna(tabla, "feedback.acierto", "acierto"), errors="coerce"
            ),
            "formato_valido": pd.to_numeric(
                _columna(tabla, "feedback.formato_valido", "formato_valido"), errors="coerce"
            ),
            "latencia_s": pd.to_numeric(
                _columna(tabla, "outputs.latencia_s", "latencia_s"), errors="coerce"
            ),
        }
    )


def wilson(aciertos: int, n: int, z: float = 1.959963984540054) -> tuple[float, float]:
    """Intervalo de confianza de Wilson para una proporción."""
    if n == 0:
        return float("nan"), float("nan")
    p = aciertos / n
    denominador = 1 + z**2 / n
    centro = (p + z**2 / (2 * n)) / denominador
    mitad = z * ((p * (1 - p) / n + z**2 / (4 * n**2)) ** 0.5) / denominador
    return max(0.0, centro - mitad), min(1.0, centro + mitad)


CATEGORIAS = CLASES + ["<inválido>"]
resumen = []
corridas = {}

for nombre in MODELOS:
    tabla = cargar_routing(nombre)
    corridas[nombre] = tabla
    y_true = tabla["dominio_esperado"]
    y_pred = tabla["dominio_predicho"].where(
        tabla["dominio_predicho"].isin(CLASES), "<inválido>"
    )
    n = len(tabla)
    aciertos = int((y_true == y_pred).sum())
    ic_bajo, ic_alto = wilson(aciertos, n)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=CLASES,
        average="macro",
        zero_division=0,
    )
    matriz = pd.crosstab(
        pd.Categorical(y_true, categories=CATEGORIAS),
        pd.Categorical(y_pred, categories=CATEGORIAS),
        dropna=False,
    )
    matriz.to_csv(
        REPORTS / f"evaluacion_routing_matriz_{nombre}.csv", encoding="utf-8-sig"
    )
    resumen.append(
        {
            "modelo": nombre,
            "modelo_id": MODELOS[nombre],
            "n": n,
            "aciertos": aciertos,
            "accuracy": round(aciertos / n, 4),
            "ic_bajo": round(ic_bajo, 4),
            "ic_alto": round(ic_alto, 4),
            "precision_macro": round(precision, 4),
            "recall_macro": round(recall, 4),
            "f1_macro": round(f1, 4),
            "pct_formato_valido": round(tabla["formato_valido"].mean(), 4),
            "latencia_mediana_s": round(tabla["latencia_s"].median(), 3),
        }
    )

resumen_df = pd.DataFrame(resumen)
resumen_df.to_csv(
    REPORTS / "evaluacion_routing_resumen.csv", index=False, encoding="utf-8-sig"
)
print(resumen_df.to_string(index=False))
for nombre in MODELOS:
    print(f"\nMatriz de confusión · {nombre}")
    print(pd.read_csv(REPORTS / f"evaluacion_routing_matriz_{nombre}.csv", index_col=0).to_string())

modelo                          modelo_id  n  aciertos  accuracy  ic_bajo  ic_alto  precision_macro  recall_macro  f1_macro  pct_formato_valido  latencia_mediana_s
gemini google_genai:gemini-3.1-flash-lite 30        29    0.9667   0.8333   0.9941           0.9697        0.9667    0.9666              1.0000               2.872
  groq           groq:openai/gpt-oss-120b 30        28    0.9333   0.7868   0.9815           0.9697        0.9333    0.9471              0.9667               0.539

Matriz de confusión · gemini
                       infraestructura_redes  soc_logs  threat_intel  <inválido>
row_0                                                                           
infraestructura_redes                     10         0             0           0
soc_logs                                   0        10             0           0
threat_intel                               0         1             9           0
<inválido>                                 0         0             0     

## 6. Comparación pareada de los dos modelos

Como ambos routers responden **las mismas 30 preguntas**, la comparación es pareada: solo cuentan los casos en que uno acierta y el otro falla. La **prueba de McNemar exacta** (`binomtest` con p = 0.5) evalúa si la diferencia observada es distinguishable. No se comparan intervalos de Wald independientes.

In [8]:
from scipy.stats import binomtest

if "corridas" not in globals() or not corridas:
    corridas = {nombre: cargar_routing(nombre) for nombre in MODELOS}

gemini = corridas["gemini"].set_index("pregunta")
groq = corridas["groq"].set_index("pregunta")
comunes = sorted(set(gemini.index) & set(groq.index))
ace_g = gemini.loc[comunes, "acierto"]
ace_q = groq.loc[comunes, "acierto"]

b = int(((ace_g == 1) & (ace_q == 0)).sum())  # Gemini acierta, Groq falla
c = int(((ace_g == 0) & (ace_q == 1)).sum())  # Groq acierta, Gemini falla
p_mcnemar = binomtest(b, b + c, 0.5).pvalue if (b + c) else 1.0

comparacion = pd.DataFrame(
    [
        {
            "n_pareado": len(comunes),
            "accuracy_gemini": round(ace_g.mean(), 4),
            "accuracy_groq": round(ace_q.mean(), 4),
            "diferencia_gemini_menos_groq": round(ace_g.mean() - ace_q.mean(), 4),
            "gemini_acierta_groq_falla": b,
            "groq_acierta_gemini_falla": c,
            "p_mcnemar": round(p_mcnemar, 4),
        }
    ]
)
ruta_comparacion = REPORTS / "evaluacion_routing_comparacion.csv"
comparacion.to_csv(ruta_comparacion, index=False, encoding="utf-8-sig")
print(comparacion.to_string(index=False))
print(f"\nDiscordantes: {b + c} de {len(comunes)} pares")

 n_pareado  accuracy_gemini  accuracy_groq  diferencia_gemini_menos_groq  gemini_acierta_groq_falla  groq_acierta_gemini_falla  p_mcnemar
        30           0.9667         0.9333                        0.0333                          1                          0        1.0

Discordantes: 1 de 30 pares


## 7. Conclusiones

Ejecución del 18/11/2026 con `temperature=0`, semilla 42, `num_repetitions=1`, `max_concurrency=2` y las mismas 30 preguntas para ambos routers:

| Router | Acierto | IC 95 % (Wilson) | F1 macro | Formato válido | Latencia mediana |
|---|---|---|---|---|---|
| **Gemini** (activo, `MODEL_1`) | **29/30 = 96,7 %** | 83,3 % – 99,4 % | 0,967 | 30/30 (100 %) | 2,87 s |
| Groq (alternativo, `MODEL_2`) | 28/30 = 93,3 % | 78,7 % – 98,2 % | 0,947 | 29/30 (96,7 %) | 0,54 s |

**El enrutamiento queda medido y validado.** Ambos modelos aciertan el 100 % de `infraestructura_redes` (10/10) y `soc_logs` (10/10). El único foco de error es `threat_intel`: la consulta *«Quiero generar un informe con reglas de detección para Wazuh a partir de un incidente»* la clasifican como `soc_logs` en los dos modelos. Es una ambigüedad real del prompt —menciona Wazuh, herramienta del dominio SOC, aunque pide inteligencia de amenazas— y es el primer candidato a corregir en `ROUTER_SYSTEM_PROMPT` (por ejemplo, aclarando que «reglas de detección» son tarea de threat intel y «consultar alertas» es SOC).

**No hay evidencia de que un modelo sea mejor enrutando.** Gemini acierta un caso más, pero ese único par discordante no fue una decisión: Groq devolvió un **429 por límite de tokens por minuto** y el pipeline lo registró como fallo sin predicción. Con `b=1` y `c=0`, el test de McNemar exacto da **`p = 1,0`**. Si se descarta esa fila por error de infraestructura, ambos modelos quedan en **28/29 aciertos (96,6 %)** sobre las preguntas donde los dos llegaron a decidir. La diferencia observada de 3,3 puntos no es significativa y no justifica cambiar el modelo activo. En cambio, Groq es unas 5 veces más rápido en la mediana (0,54 s vs 2,87 s), con una cola mucho más corta: Gemini tuvo un caso de 160,97 s y varios por encima de 13 s.

**Limitaciones.** La muestra es pequeña (n = 30) y los intervalos de Wilson son anchos: con un solo error, el límite inferior ya cae al 83 %. Hay una sola corrida por modelo, aunque con temperatura 0 y semilla fija. Las consultas fueron redactadas para este experimento, así que el resultado depende del diseño de prompts y no representa tráfico real de analistas. El dominio `threat_intel` tiene diez preguntas heterogéneas, por lo que un solo caso ambiguo pesa mucho en su recall (9/10). Finalmente, `_extract_domain()` tiene un fallback silencioso a `threat_intel`: en esta corrida no enmascaró ninguna salida inválida de Gemini, pero por eso se midió `formato_valido` por separado; conviene vigilarlo si el router se usa con otros modelos. Los dominios `web_apis` y `mobile` están comentados en `DOMAINS` y quedan fuera de alcance.

**Artefactos.** `reports/evaluacion_routing_resumen.csv`, `reports/evaluacion_routing_matriz_gemini.csv`, `reports/evaluacion_routing_matriz_groq.csv`, `reports/evaluacion_routing_comparacion.csv` y los dos exports crudos `reports/evaluacion_routing_{gemini,groq}_raw.csv`. Los experimentos `Capazeta_Routing_Gemini` y `Capazeta_Routing_Groq` quedan trackeados en LangSmith sobre el dataset `Evaluacion_Routing_Capazeta`.